# LeetCode 3585 — Find Weighted Median Node in Tree bằng Python

- Link: https://leetcode.com/problems/find-weighted-median-node-in-tree/
- Độ khó: **Hard**
- Chủ đề: **Cây có trọng số**, **LCA bằng binary lifting**, **khoảng cách tiền tố từ gốc**, **tìm kiếm nhị phân bằng bước nhảy lũy thừa 2**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMedian(self, n: int, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Tính **khoảng cách có trọng số từ gốc** `D[x]` và **độ sâu** `dep[x]` bằng BFS; từ đó trọng số đường đi `u → v` là `D[u] + D[v] − 2·D[lca]`.
2. Cài đặt **binary lifting** (`up[j][x]` = tổ tiên thứ `2^j` của `x`) và dùng nó để tìm **LCA** trong `O(log n)`.
3. Tách đường đi `u → v` thành hai nửa `u ↑ lca` và `lca ↓ v`, xác định "trung vị có trọng số" nằm ở nửa nào chỉ bằng một phép so sánh.
4. Dùng binary lifting như một **tìm kiếm nhị phân trên tổ tiên**: nhảy các bước `2^j` giảm dần trong khi điều kiện còn thỏa.
5. So sánh tránh số thực: `sum ≥ tot / 2` ⇔ `2·sum ≥ tot`.

## 2. Đề bài & ví dụ

Cây vô hướng có trọng số, gốc `0`, `n` đỉnh; `edges[i] = [u, v, w]`.

**Đỉnh trung vị có trọng số** của đường đi từ `u` tới `v` là đỉnh `x` **đầu tiên** trên đường (đi từ `u` về phía `v`) sao cho tổng trọng số từ `u` tới `x` **≥ một nửa** tổng trọng số cả đường.

Với mỗi truy vấn `[u, v]`, trả về đỉnh trung vị.

**Ví dụ 1:** `n=2, edges=[[0,1,7]], queries=[[1,0],[0,1]]` → `[0,1]`.

**Ví dụ 2:** `n=3, edges=[[0,1,2],[2,0,4]], queries=[[0,1],[2,0],[1,2]]` → `[1,0,2]`. Với `[1,2]`: đường `1→0→2`, trọng số `[2,4]`, tổng 6, nửa là 3; tại 0 tổng là 2 < 3; tại 2 tổng là 6 ≥ 3 ⇒ đáp án 2.

**Ví dụ 3:** `n=5, edges=[[0,1,2],[0,2,5],[1,3,1],[2,4,3]], queries=[[3,4],[1,2]]` → `[2,2]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `1 ≤ w ≤ 10^9`, `1 ≤ len(queries) ≤ 10^5`.

Ghi chú: nếu `u = v` thì tổng bằng 0, ngay đỉnh `u` đã có tổng `0 ≥ 0` ⇒ đáp án `u`.

In [1]:
from typing import List
from collections import deque
examples = [
    (2, [[0, 1, 7]], [[1, 0], [0, 1]], [0, 1]),
    (3, [[0, 1, 2], [2, 0, 4]], [[0, 1], [2, 0], [1, 2]], [1, 0, 2]),
    (5, [[0, 1, 2], [0, 2, 5], [1, 3, 1], [2, 4, 3]], [[3, 4], [1, 2]], [2, 2]),
]
for n, e, q, exp in examples:
    print(n, e, q, '-> kỳ vọng', exp)

2 [[0, 1, 7]] [[1, 0], [0, 1]] -> kỳ vọng [0, 1]
3 [[0, 1, 2], [2, 0, 4]] [[0, 1], [2, 0], [1, 2]] -> kỳ vọng [1, 0, 2]
5 [[0, 1, 2], [0, 2, 5], [1, 3, 1], [2, 4, 3]] [[3, 4], [1, 2]] -> kỳ vọng [2, 2]


## 3. Kiến thức nền

### 3.1. Danh sách kề và BFS từ gốc

Từ danh sách cạnh vô hướng dựng `adj[u] = [(v, w), ...]`. BFS từ gốc 0 cho: `par[v]` (cha), `dep[v]` (số cạnh tới gốc), `D[v]` (tổng trọng số tới gốc).

In [2]:
def bfs_root(n, edges):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    par, dep, D = [-1] * n, [0] * n, [0] * n
    seen = [False] * n
    seen[0] = True
    order = [0]
    for u in order:
        for v, w in adj[u]:
            if not seen[v]:
                seen[v] = True
                par[v], dep[v], D[v] = u, dep[u] + 1, D[u] + w
                order.append(v)
    return par, dep, D

par, dep, D = bfs_root(5, [[0, 1, 2], [0, 2, 5], [1, 3, 1], [2, 4, 3]])
print('par =', par, '| dep =', dep, '| D =', D)

par = [-1, 0, 0, 1, 2] | dep = [0, 1, 1, 2, 2] | D = [0, 2, 5, 3, 8]


### 3.2. Binary lifting

`up[0][x] = par[x]` (gốc trỏ về chính nó), `up[j][x] = up[j-1][ up[j-1][x] ]`. Muốn lên `k` bậc: phân tích `k` thành tổng lũy thừa 2 và nhảy tương ứng — `O(log n)`.

In [3]:
def build_up(par):
    n = len(par)
    LOG = max(1, (n - 1).bit_length())
    up = [[p if p >= 0 else 0 for p in par]]
    for _ in range(1, LOG):
        prev = up[-1]
        up.append([prev[prev[x]] for x in range(n)])
    return up

def kth_ancestor(up, x, k):
    j = 0
    while k:
        if k & 1:
            x = up[j][x]
        k >>= 1
        j += 1
    return x

up = build_up(par)
print('LOG =', len(up), '| up[0] =', up[0], '| up[1] =', up[1])
print('tổ tiên bậc 2 của đỉnh 4:', kth_ancestor(up, 4, 2))

LOG = 3 | up[0] = [0, 0, 0, 1, 2] | up[1] = [0, 0, 0, 0, 0]
tổ tiên bậc 2 của đỉnh 4: 0


### 3.3. LCA (tổ tiên chung thấp nhất)

1. Đưa đỉnh sâu hơn lên cùng độ sâu.
2. Nếu trùng nhau ⇒ đó là LCA.
3. Ngược lại nhảy cả hai lên với `j` giảm dần **khi tổ tiên `2^j` của chúng còn khác nhau**; cuối cùng LCA là cha của chúng.

In [4]:
def lca(up, dep, a, b):
    if dep[a] < dep[b]:
        a, b = b, a
    a = kth_ancestor(up, a, dep[a] - dep[b])
    if a == b:
        return a
    for j in range(len(up) - 1, -1, -1):
        if up[j][a] != up[j][b]:
            a, b = up[j][a], up[j][b]
    return up[0][a]

print('lca(3, 4) =', lca(up, dep, 3, 4), '| lca(3, 1) =', lca(up, dep, 3, 1))

lca(3, 4) = 0 | lca(3, 1) = 1


### 3.4. So sánh "≥ một nửa" không dùng số thực

`s ≥ tot / 2` ⇔ `2·s ≥ tot` (với số nguyên). Tránh hoàn toàn chia và sai số dấu phẩy động; Python số nguyên không tràn nên `2·10^14` vẫn an toàn.

In [5]:
tot = 11
for s in [3, 5, 6, 8]:
    print(s, '>= tot/2 ?', s >= tot / 2, '| 2*s >= tot ?', 2 * s >= tot)

3 >= tot/2 ? False | 2*s >= tot ? False
5 >= tot/2 ? False | 2*s >= tot ? False
6 >= tot/2 ? True | 2*s >= tot ? True
8 >= tot/2 ? True | 2*s >= tot ? True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đi từ `u` tới `v`, tổng trọng số tích lũy **tăng dần**. Ta cần đỉnh đầu tiên mà tổng tích lũy vượt (hoặc bằng) nửa tổng ⇒ đây là **tìm kiếm nhị phân trên đường đi**.

Đường đi `u → v` = `u ↑ l` (đi lên tới LCA `l`) rồi `l ↓ v` (đi xuống). Tổng tích lũy:

- tại đỉnh `x` trên nửa đầu (tổ tiên của `u`, sâu hơn hoặc bằng `l`): `D[u] − D[x]`;
- tại đỉnh `x` trên nửa sau (tổ tiên của `v`, sâu hơn `l`): `(D[u] − D[l]) + (D[x] − D[l])`.

Tính tay ví dụ 3, truy vấn `[3, 4]`: `D = [0, 2, 5, 3, 8]`, `l = 0`, `tot = 3 + 8 − 0 = 11`.

| đỉnh trên đường | tổng từ 3 | 2·tổng ≥ 11? |
|---|---|---|
| 3 | 0 | không |
| 1 | 1 | không |
| 0 (= l) | 3 | không |
| 2 | 3 + 5 = 8 | **có** → đáp án 2 |
| 4 | 11 | có |

In [6]:
def path_nodes(par, dep, u, v):
    """Trả về danh sách đỉnh trên đường u -> v (đi theo cha)."""
    left, right = [u], [v]
    a, b = u, v
    while dep[a] > dep[b]:
        a = par[a]; left.append(a)
    while dep[b] > dep[a]:
        b = par[b]; right.append(b)
    while a != b:
        a, b = par[a], par[b]
        left.append(a); right.append(b)
    return left + right[-2::-1]

par, dep, D = bfs_root(5, [[0, 1, 2], [0, 2, 5], [1, 3, 1], [2, 4, 3]])
p = path_nodes(par, dep, 3, 4)
print('đường 3 -> 4:', p)
for x in p:
    l = 0
    s = (D[3] - D[x]) if x in (3, 1, 0) else (D[3] - D[l]) + (D[x] - D[l])
    print('  đỉnh', x, 'tổng tích lũy', s)

đường 3 -> 4: [3, 1, 0, 2, 4]
  đỉnh 3 tổng tích lũy 0
  đỉnh 1 tổng tích lũy 1
  đỉnh 0 tổng tích lũy 3
  đỉnh 2 tổng tích lũy 8
  đỉnh 4 tổng tích lũy 11


## 5. Cách 1 — Brute Force: BFS từ `u` để dựng đường đi rồi quét

Với mỗi truy vấn: BFS từ `u` lưu cha, lần ngược từ `v` để có đường đi và trọng số các cạnh, cộng dồn và dừng ở đỉnh đầu tiên có `2·s ≥ tot`.

In [7]:
def median_brute(n, edges, queries):
    adj = [[] for _ in range(n)]
    for a, b, w in edges:
        adj[a].append((b, w))
        adj[b].append((a, w))
    out = []
    for u, v in queries:
        prev = {u: (None, 0)}
        dq = deque([u])
        while dq:
            x = dq.popleft()
            for y, w in adj[x]:
                if y not in prev:
                    prev[y] = (x, w)
                    dq.append(y)
        path, ws = [v], []
        while path[-1] != u:
            p, w = prev[path[-1]]
            ws.append(w)
            path.append(p)
        path.reverse(); ws.reverse()
        tot, s = sum(ws), 0
        for i, x in enumerate(path):
            if i:
                s += ws[i - 1]
            if 2 * s >= tot:
                out.append(x)
                break
    return out

for n, e, q, exp in examples:
    print(median_brute(n, e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[1, 0, 2] | kỳ vọng [1, 0, 2]
[2, 2] | kỳ vọng [2, 2]


**Độ phức tạp:** `O(n)` mỗi truy vấn ⇒ `O(n · q)` = `10^10` với giới hạn đề.

**Nút thắt:** (1) phải BFS lại cả cây; (2) phải đi từng đỉnh dọc đường đi. Ta cần: tính tổng tích lũy tại một đỉnh bất kỳ trong `O(1)` (nhờ `D`), và nhảy dọc đường đi theo lũy thừa 2.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — tổng đường đi trong `O(1)`:** `tot = D[u] + D[v] − 2·D[l]`.

**Bước 2 — xác định nửa:** tổng tích lũy tại `l` là `A = D[u] − D[l]`.

- **Nếu** `2·A ≥ tot` **thì** trung vị nằm trên đoạn `u ↑ l` (có thể chính là `l`).
- **Ngược lại** trung vị nằm trên đoạn `l ↓ v`, sâu hơn `l` hẳn.

**Bước 3 — nhảy nhị phân (tính đơn điệu):**

- Nửa đầu: điều kiện `2·(D[u] − D[x]) ≥ tot` đúng với các tổ tiên **cao**, sai với các đỉnh gần `u`. Từ `u`, nhảy lên `2^j` (j giảm dần) **nếu đỉnh đích vẫn còn sai điều kiện** (và không vượt quá `l`). Khi dừng ở `x` (đỉnh cao nhất còn sai) thì đáp án là `par[x]`. Riêng khi `tot = 0` (u = v) đáp án là `u`.
- Nửa sau: điều kiện `2·(A + D[x] − D[l]) ≥ tot` đúng với các đỉnh **sâu** (gần `v`). Từ `v`, nhảy lên `2^j` **nếu đỉnh đích còn đúng điều kiện và vẫn sâu hơn `l`**. Khi dừng, đỉnh hiện tại là đỉnh cao nhất đúng điều kiện ⇒ đó là đáp án (gần `u` nhất).

Mỗi truy vấn: LCA `O(log n)` + một lần nhảy `O(log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đi theo cha (không BFS lại), quét đường đi

Tiền xử lý `par, dep, D` một lần. Mỗi truy vấn dựng đường đi bằng cách leo cha (hàm `path_nodes`), tổng tích lũy tính bằng `D`. Đỡ chi phí BFS nhưng vẫn `O(độ dài đường)` mỗi truy vấn.

In [8]:
def median_climb(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    out = []
    for u, v in queries:
        path = path_nodes(par, dep, u, v)
        # LCA là đỉnh có độ sâu nhỏ nhất trên đường
        l = min(path, key=dep.__getitem__)
        tot = D[u] + D[v] - 2 * D[l]
        seen_l = False
        for x in path:
            if not seen_l:
                s = D[u] - D[x]
            else:
                s = D[u] - D[l] + D[x] - D[l]
            if x == l:
                seen_l = True
            if 2 * s >= tot:
                out.append(x)
                break
    return out

for n, e, q, exp in examples:
    print(median_climb(n, e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[1, 0, 2] | kỳ vọng [1, 0, 2]
[2, 2] | kỳ vọng [2, 2]


**Độ phức tạp:** `O(n + q · h)` với `h` là chiều cao cây — vẫn `O(n q)` trên cây đường thẳng.

### 7.2. Cách 3 — Binary lifting + tìm kiếm nhị phân trên số bước (O(log² n))

Ý tưởng dễ nghĩ nhất sau khi có `kth_ancestor`: đường đi có `len = dep[u] + dep[v] − 2·dep[l]` cạnh; đỉnh thứ `i` trên đường (tính từ `u`) lấy được bằng `kth_ancestor`. Tổng tích lũy tăng theo `i` ⇒ **binary search `i`**. Mỗi bước binary search tốn `O(log n)` ⇒ `O(log² n)` mỗi truy vấn.

In [9]:
def median_bs(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    out = []
    for u, v in queries:
        l = lca(up, dep, u, v)
        du, dv = dep[u] - dep[l], dep[v] - dep[l]
        tot = D[u] + D[v] - 2 * D[l]

        def node_at(i):                       # đỉnh thứ i trên đường (0 = u)
            if i <= du:
                return kth_ancestor(up, u, i)
            return kth_ancestor(up, v, du + dv - i)

        def acc(i):
            x = node_at(i)
            return D[u] - D[x] if i <= du else D[u] - 2 * D[l] + D[x]

        lo, hi = 0, du + dv
        while lo < hi:
            mid = (lo + hi) // 2
            if 2 * acc(mid) >= tot:
                hi = mid
            else:
                lo = mid + 1
        out.append(node_at(lo))
    return out

for n, e, q, exp in examples:
    print(median_bs(n, e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[1, 0, 2] | kỳ vọng [1, 0, 2]
[2, 2] | kỳ vọng [2, 2]


**Độ phức tạp:** tiền xử lý `O(n log n)`, mỗi truy vấn `O(log² n)`.

### 7.3. Cách 4 — Binary lifting nhảy trực tiếp (lời giải chính, O(log n) mỗi truy vấn)

Thay vì binary search số bước rồi mới nhảy, ta **nhảy trực tiếp** như mô tả ở mục 6 — gộp hai tầng log thành một.

In [10]:
def median_lifting(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    LOG = len(up)
    out = []
    for u, v in queries:
        l = lca(up, dep, u, v)
        tot = D[u] + D[v] - 2 * D[l]
        if tot == 0:
            out.append(u)
            continue
        if 2 * (D[u] - D[l]) >= tot:                 # nửa u ↑ l
            x = u
            for j in range(LOG - 1, -1, -1):
                y = up[j][x]
                if dep[y] >= dep[l] and 2 * (D[u] - D[y]) < tot:
                    x = y                            # y vẫn chưa đủ nửa -> nhảy
            out.append(par[x])
        else:                                        # nửa l ↓ v
            x = v
            base = D[u] - 2 * D[l]
            for j in range(LOG - 1, -1, -1):
                y = up[j][x]
                if dep[y] > dep[l] and 2 * (base + D[y]) >= tot:
                    x = y                            # y vẫn đủ nửa -> nhảy lên
            out.append(x)
    return out

for n, e, q, exp in examples:
    print(median_lifting(n, e, q), '| kỳ vọng', exp)

[0, 1] | kỳ vọng [0, 1]
[1, 0, 2] | kỳ vọng [1, 0, 2]
[2, 2] | kỳ vọng [2, 2]


Vì sao ở nửa đầu luôn có `par[x]` hợp lệ? Vì `2·(D[u] − D[l]) ≥ tot > 0` nên `l` thỏa điều kiện, còn `u` thì không (`0 < tot`); `x` là đỉnh cao nhất **không** thỏa, nằm thấp hơn `l` ⇒ `par[x]` nằm trên đoạn và là đỉnh đầu tiên thỏa.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 3 với hai truy vấn `[3, 4]` (trung vị ở nửa sau) và `[4, 3]` (đảo chiều, trung vị ở nửa đầu).

In [11]:
def trace_query(n, edges, u, v):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    l = lca(up, dep, u, v)
    tot = D[u] + D[v] - 2 * D[l]
    A = D[u] - D[l]
    print(f"Truy vấn [{u}, {v}]: lca = {l}, D = {D}, tot = {tot}, tổng tại lca = {A}")
    if tot == 0:
        print('  u = v -> đáp án', u); return u
    if 2 * A >= tot:
        print('  2·A >= tot -> trung vị ở nửa u ↑ lca')
        x = u
        for j in range(len(up) - 1, -1, -1):
            y = up[j][x]
            cond = dep[y] >= dep[l] and 2 * (D[u] - D[y]) < tot
            print(f"   j={j}: thử nhảy {x} -> {y} (2·tổng={2*(D[u]-D[y])}) -> {'NHẢY' if cond else 'không'}")
            if cond:
                x = y
        print('  dừng tại', x, '-> đáp án = cha của nó =', par[x]); return par[x]
    print('  2·A < tot -> trung vị ở nửa lca ↓ v')
    x = v
    base = D[u] - 2 * D[l]
    for j in range(len(up) - 1, -1, -1):
        y = up[j][x]
        cond = dep[y] > dep[l] and 2 * (base + D[y]) >= tot
        print(f"   j={j}: thử nhảy {x} -> {y} (2·tổng={2*(base+D[y])}) -> {'NHẢY' if cond else 'không'}")
        if cond:
            x = y
    print('  đáp án =', x); return x

E3 = [[0, 1, 2], [0, 2, 5], [1, 3, 1], [2, 4, 3]]
trace_query(5, E3, 3, 4)
print()
trace_query(5, E3, 4, 3)

Truy vấn [3, 4]: lca = 0, D = [0, 2, 5, 3, 8], tot = 11, tổng tại lca = 3
  2·A < tot -> trung vị ở nửa lca ↓ v
   j=2: thử nhảy 4 -> 0 (2·tổng=6) -> không
   j=1: thử nhảy 4 -> 0 (2·tổng=6) -> không
   j=0: thử nhảy 4 -> 2 (2·tổng=16) -> NHẢY
  đáp án = 2

Truy vấn [4, 3]: lca = 0, D = [0, 2, 5, 3, 8], tot = 11, tổng tại lca = 8
  2·A >= tot -> trung vị ở nửa u ↑ lca
   j=2: thử nhảy 4 -> 0 (2·tổng=16) -> không
   j=1: thử nhảy 4 -> 0 (2·tổng=16) -> không
   j=0: thử nhảy 4 -> 2 (2·tổng=6) -> NHẢY
  dừng tại 2 -> đáp án = cha của nó = 0


0

Kiểm tra tay `[4, 3]`: đường `4→2→0→1→3`, trọng số `[3,5,2,1]`, tổng 11. Tích lũy: 0, 3, 8 ⇒ đỉnh 0 có 8 ≥ 5.5 ⇒ đáp án **0**. Trace phải ra 0.

Trace thêm trên một cây đường thẳng dài để thấy các bước nhảy lớn:

In [12]:
chain = [[i, i + 1, 1] for i in range(15)]           # 16 đỉnh, mọi cạnh trọng số 1
trace_query(16, chain, 15, 0)

Truy vấn [15, 0]: lca = 0, D = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], tot = 15, tổng tại lca = 15
  2·A >= tot -> trung vị ở nửa u ↑ lca
   j=3: thử nhảy 15 -> 7 (2·tổng=16) -> không
   j=2: thử nhảy 15 -> 11 (2·tổng=8) -> NHẢY
   j=1: thử nhảy 11 -> 9 (2·tổng=12) -> NHẢY
   j=0: thử nhảy 9 -> 8 (2·tổng=14) -> NHẢY
  dừng tại 8 -> đáp án = cha của nó = 7


7

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def findMedian(self, n: int, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n)]
        for a, b, w in edges:
            adj[a].append((b, w))
            adj[b].append((a, w))
        par, dep, D = [0] * n, [0] * n, [0] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for x in order:
            for y, w in adj[x]:
                if not seen[y]:
                    seen[y] = True
                    par[y], dep[y], D[y] = x, dep[x] + 1, D[x] + w
                    order.append(y)
        LOG = max(1, (n - 1).bit_length())
        up = [par]                       # par[0] = 0: gốc trỏ về chính nó
        for _ in range(1, LOG):
            pr = up[-1]
            up.append([pr[pr[x]] for x in range(n)])

        res = []
        for u, v in queries:
            a, b = u, v
            if dep[a] < dep[b]:
                a, b = b, a
            diff, j = dep[a] - dep[b], 0
            while diff:
                if diff & 1:
                    a = up[j][a]
                diff >>= 1
                j += 1
            if a != b:
                for j in range(LOG - 1, -1, -1):
                    if up[j][a] != up[j][b]:
                        a, b = up[j][a], up[j][b]
                a = par[a]
            l = a
            tot = D[u] + D[v] - 2 * D[l]
            if tot == 0:
                res.append(u)
            elif 2 * (D[u] - D[l]) >= tot:
                x = u
                for j in range(LOG - 1, -1, -1):
                    y = up[j][x]
                    if dep[y] >= dep[l] and 2 * (D[u] - D[y]) < tot:
                        x = y
                res.append(par[x])
            else:
                x, base = v, D[u] - 2 * D[l]
                for j in range(LOG - 1, -1, -1):
                    y = up[j][x]
                    if dep[y] > dep[l] and 2 * (base + D[y]) >= tot:
                        x = y
                res.append(x)
        return res

for n, e, q, exp in examples:
    got = Solution().findMedian(n, e, q)
    print(got, 'OK' if got == exp else 'SAI')

[0, 1] OK
[1, 0, 2] OK
[2, 2] OK


## 10. Vì sao thuật toán đúng

1. **Tổng tích lũy đơn điệu:** trọng số `w ≥ 1` nên đi dọc đường `u → v`, tổng tích lũy **tăng ngặt**. Tập đỉnh thỏa `2·s ≥ tot` là một **hậu tố** của đường đi; ta cần phần tử đầu của hậu tố đó.
2. **Chọn nửa đúng:** `l` là đỉnh cuối của nửa đầu. Nếu `l` thỏa thì hậu tố bắt đầu tại hoặc trước `l` ⇒ trong nửa đầu; nếu không thì bắt đầu sau `l` ⇒ trong nửa sau.
3. **Nhảy ở nửa đầu:** trên dãy tổ tiên của `u` (từ `u` lên `l`), điều kiện "chưa thỏa" là một **tiền tố**. Phép nhảy `2^j` giảm dần (chỉ nhảy khi đích còn chưa thỏa và không vượt `l`) dừng ở đỉnh cuối cùng của tiền tố đó — chuẩn mực của binary lifting (giống tìm tổ tiên xa nhất thỏa điều kiện đơn điệu). Đỉnh kế tiếp (cha) là đỉnh đầu tiên thỏa.
4. **Nhảy ở nửa sau:** trên dãy tổ tiên của `v` ngặt dưới `l`, đi từ `v` lên, điều kiện "thỏa" là tiền tố (gần `v`). Dừng ở đỉnh cao nhất còn thỏa = đỉnh đầu tiên thỏa khi đi theo chiều `u → v`.
5. **LCA đúng** theo thuật toán chuẩn; `tot = D[u] + D[v] − 2·D[l]` vì phần đường từ `l` lên gốc bị đếm hai lần.

## 11. Phân tích Time / Space Complexity

Gọi `n` số đỉnh, `q` số truy vấn, `LOG = ⌈log2 n⌉ ≈ 17`.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. BFS mỗi truy vấn | — | `O(n)` | `O(n)` |
| 2. Leo cha | `O(n)` | `O(h)` | `O(n)` |
| 3. Lifting + binary search | `O(n log n)` | `O(log² n)` | `O(n log n)` |
| 4. Lifting nhảy trực tiếp | `O(n log n)` | `O(log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'climb': median_climb,
    'bs': median_bs,
    'lifting': median_lifting,
    'submit': lambda n, e, q: Solution().findMedian(n, e, q),
}
edge = [
    (2, [[1, 0, 5]], [[0, 0], [1, 1]], [0, 1]),              # u = v
    (3, [[0, 1, 3], [1, 2, 3]], [[0, 2], [2, 0]], [1, 1]),  # đúng một nửa: 2·3 >= 6 tại đỉnh giữa
    (4, [[0, 1, 1], [0, 2, 1], [0, 3, 100]], [[1, 3], [3, 1], [1, 2]], [3, 0, 0]),
    (3, [[0, 1, 10**9], [1, 2, 10**9]], [[2, 0]], [1]),     # trọng số lớn
]
for n, e, q, exp in examples + edge:
    for name, f in impls.items():
        got = f(n, e, q)
        assert got == exp, (name, n, e, q, got, exp)
    assert median_brute(n, e, q) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (đánh số đỉnh ngẫu nhiên, chiều cạnh ngẫu nhiên)

In [15]:
import random
random.seed(3585)

def rand_tree(n, wmax=6):
    perm = list(range(n))
    random.shuffle(perm)
    perm[perm.index(0)], perm[0] = perm[0], 0       # đảm bảo cả cây có đỉnh 0
    es = []
    for i in range(1, n):
        a, b = perm[random.randrange(i)], perm[i]
        if random.random() < 0.5:
            a, b = b, a
        es.append([a, b, random.randint(1, wmax)])
    return es

cnt = 0
for _ in range(500):
    n = random.randint(2, 14)
    e = rand_tree(n)
    q = [[random.randrange(n), random.randrange(n)] for _ in range(12)]
    exp = median_brute(n, e, q)
    for name, f in impls.items():
        assert f(n, e, q) == exp, (name, n, e, q)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 500 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [16]:
import time
random.seed(10)
n = 10**5
qs = [[random.randrange(n), random.randrange(n)] for _ in range(n)]
for name, e in [('cây ngẫu nhiên', rand_tree(n, 10**9)),
                ('đường thẳng', [[i, i + 1, random.randint(1, 10**9)] for i in range(n - 1)])]:
    t = time.time()
    res = Solution().findMedian(n, e, qs)
    print(f"{name}: {time.time() - t:.2f}s, 5 kết quả đầu {res[:5]}")

cây ngẫu nhiên: 1.16s, 5 kết quả đầu [72242, 16603, 47050, 2248, 66686]


đường thẳng: 1.80s, 5 kết quả đầu [39632, 59739, 38882, 43841, 50396]


Khoảng 1–2 giây cho `10^5` truy vấn trong Python. Cách 3 (`log²`) chậm hơn cỡ 10 lần nên không đưa vào ca lớn.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh với `tot // 2` (chia nguyên)

In [17]:
def wrong_floor_half(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    out = []
    for u, v in queries:
        path = path_nodes(par, dep, u, v)
        l = min(path, key=dep.__getitem__)
        tot = D[u] + D[v] - 2 * D[l]
        seen_l = False
        for x in path:
            s = D[u] - D[x] if not seen_l else D[u] - 2 * D[l] + D[x]
            seen_l = seen_l or x == l
            if s >= tot // 2:            # SAI: 7 // 2 = 3 < 3.5
                out.append(x)
                break
    return out

e, q = [[0, 1, 3], [1, 2, 4]], [[0, 2]]
print('sai :', wrong_floor_half(3, e, q))    # [1] vì 3 >= 3
print('đúng:', Solution().findMedian(3, e, q))  # [2] vì 3 < 3.5

sai : [1]
đúng: [2]


**Vì sao sai:** `tot / 2 = 3.5`, tổng 3 **chưa** đạt. `tot // 2` làm tròn xuống thành 3 nên chấp nhận sai. Dùng `2·s ≥ tot`.

### Lỗi 2 — Luôn tìm trên nửa `u ↑ lca`, quên nửa `lca ↓ v`

In [18]:
def wrong_only_u_side(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    out = []
    for u, v in queries:
        l = lca(up, dep, u, v)
        tot = D[u] + D[v] - 2 * D[l]
        x = u
        while x != l and 2 * (D[u] - D[x]) < tot:
            x = par[x]
        out.append(x)                    # SAI: dừng ở lca dù chưa đủ nửa
    return out

print('sai :', wrong_only_u_side(5, E3, [[3, 4]]))   # [0]
print('đúng:', Solution().findMedian(5, E3, [[3, 4]]))  # [2]

sai : [0]
đúng: [2]


**Vì sao sai:** khi `2·(D[u] − D[l]) < tot`, trung vị nằm ở **phía `v`**.

### Lỗi 3 — Ở nửa đầu trả về `x` thay vì `par[x]` (lệch một đỉnh)

In [19]:
def wrong_off_by_one(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    out = []
    for u, v in queries:
        l = lca(up, dep, u, v)
        tot = D[u] + D[v] - 2 * D[l]
        if tot == 0 or 2 * (D[u] - D[l]) < tot:
            out.append(median_lifting(n, edges, [[u, v]])[0])
            continue
        x = u
        for j in range(len(up) - 1, -1, -1):
            y = up[j][x]
            if dep[y] >= dep[l] and 2 * (D[u] - D[y]) < tot:
                x = y
        out.append(x)                    # SAI: x là đỉnh cuối cùng CHƯA đủ nửa
    return out

print('sai :', wrong_off_by_one(5, E3, [[4, 3]]))     # [2]
print('đúng:', Solution().findMedian(5, E3, [[4, 3]]))  # [0]

sai : [2]
đúng: [0]


**Vì sao sai:** vòng nhảy dừng ở đỉnh **cao nhất chưa thỏa** điều kiện; đỉnh đầu tiên thỏa là **cha** của nó. Hai cách nhảy (nửa đầu: "nhảy khi chưa thỏa", nửa sau: "nhảy khi còn thỏa") cho hai kiểu kết quả khác nhau — cần nhớ rõ.

Ghi chú: các ràng buộc độ sâu `dep[y] ≥ dep[l]` (nửa đầu) và `dep[y] > dep[l]` (nửa sau) thực ra không bao giờ bị vi phạm nhờ tính đơn điệu (đỉnh ở trên `l` luôn làm điều kiện nhảy sai), nhưng giữ chúng giúp code dễ đọc và an toàn khi sửa đổi.

### Lỗi 4 — Coi `edges[i] = [u, v, w]` là cạnh có hướng cha → con

In [20]:
def wrong_directed(n, edges, queries):
    par = [-1] * n
    D = [0] * n
    dep = [0] * n
    children = [[] for _ in range(n)]
    for u, v, w in edges:
        children[u].append((v, w))   # SAI: giả sử u là cha của v
    order = [0]
    for u in order:
        for v, w in children[u]:
            par[v], D[v], dep[v] = u, D[u] + w, dep[u] + 1
            order.append(v)
    if len(order) < n:
        return 'không duyệt được hết cây!'
    return median_climb(n, edges, queries)

print('sai :', wrong_directed(3, [[0, 1, 2], [2, 0, 4]], [[1, 2]]))
print('đúng:', Solution().findMedian(3, [[0, 1, 2], [2, 0, 4]], [[1, 2]]))

sai : không duyệt được hết cây!
đúng: [2]


**Vì sao sai:** cạnh là **vô hướng**, ví dụ 2 có cạnh `[2, 0, 4]` với cha là 0. Phải dựng danh sách kề hai chiều và BFS từ gốc.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Truy vấn trên **đường đi** giữa hai đỉnh của cây" ⇒ **LCA** + giá trị tiền tố từ gốc.
- "Đỉnh **đầu tiên** trên đường thỏa điều kiện đơn điệu" ⇒ **binary lifting** nhảy `2^j` giảm dần.
- Tách đường thành hai đoạn dọc (`u ↑ l`, `l ↓ v`) và xử lý riêng — mỗi đoạn là dãy tổ tiên của một đỉnh.
- So sánh "nửa" ⇒ nhân 2 để giữ số nguyên.
- `n = 10^5` trong Python: dựng bảng `up` bằng list comprehension, tránh đệ quy.

## 15. Biến thể của bài

1. Trung vị theo **số đỉnh** (không trọng số) ⇒ đỉnh thứ `⌈len/2⌉` trên đường ⇒ chỉ cần `kth_ancestor`.
2. Tìm đỉnh đầu tiên mà tổng vượt **tỉ lệ p** của tổng (`p` bất kỳ) ⇒ thay `2·s ≥ tot` bằng `den·s ≥ num·tot`.
3. Tìm đỉnh trên đường có **khoảng cách tới u gần với tot/2 nhất** (điểm "giữa" thật sự) ⇒ xét đỉnh trung vị và đỉnh liền trước.
4. Có **cập nhật trọng số cạnh** ⇒ `D` thay đổi theo cây con ⇒ Euler tour + Fenwick (cập nhật đoạn, truy vấn điểm).
5. Bài liên quan: LC 1483 *Kth Ancestor of a Tree Node*, LC 2846 *Minimum Edge Weight Equilibrium Queries in a Tree*.

In [21]:
# Biến thể 2: tỉ lệ p = num/den
def first_ratio(n, edges, queries, num, den):
    par, dep, D = bfs_root(n, edges)
    out = []
    for u, v in queries:
        path = path_nodes(par, dep, u, v)
        l = min(path, key=dep.__getitem__)
        tot = D[u] + D[v] - 2 * D[l]
        seen_l = False
        for x in path:
            s = D[u] - D[x] if not seen_l else D[u] - 2 * D[l] + D[x]
            seen_l = seen_l or x == l
            if den * s >= num * tot:
                out.append(x)
                break
    return out

print('p = 1/2:', first_ratio(5, E3, [[3, 4]], 1, 2), '| p = 1/4:', first_ratio(5, E3, [[3, 4]], 1, 4))

p = 1/2: [2] | p = 1/4: [0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | BFS + quét đường | `O(nq)` | `O(n)` | Oracle kiểm thử |
| 2 | Leo cha + `D` | `O(n + qh)` | `O(n)` | Cây nông |
| 3 | Lifting + binary search số bước | `O((n + q) log² n)` | `O(n log n)` | Dễ nghĩ, dễ đúng |
| 4 | Lifting nhảy trực tiếp | `O((n + q) log n)` | `O(n log n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** cây ví dụ 3, truy vấn `[[2, 3], [4, 4], [0, 4]]`.

**Bài 2 (tính tay):** đường thẳng `0-1-2-3` với trọng số `[1, 1, 10]`, truy vấn `[0, 3]` và `[3, 0]`.

**Bài 3 (điền TODO):** viết `kth_on_path(u, v, i)` trả về đỉnh thứ `i` trên đường (0 = `u`) bằng binary lifting.

**Bài 4 (tự viết lại):** biến thể 1 — trung vị theo **số cạnh**: đỉnh đầu tiên mà số cạnh đã đi `≥ len/2`.

**Bài 5 (mini challenge):** biến thể 3 — trả về đỉnh trên đường có `|s − tot/2|` nhỏ nhất (nếu hòa chọn đỉnh gần `u` hơn).

In [22]:
def kth_on_path_todo(up, dep, u, v, i):
    # TODO: tìm lca, nếu i <= dep[u]-dep[l] thì leo từ u, ngược lại leo từ v
    raise NotImplementedError

def median_by_edges_todo(n, edges, queries):
    # TODO
    raise NotImplementedError

def closest_half_todo(n, edges, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[2,3]`: đường `2→0→1→3`, trọng số `[5,2,1]`, tổng 8. (2) Tại 0 đã có 5 ≥ 4. (3) Đáp án `[0, 4, 2]` (`[0,4]`: tại 2 có 5 ≥ 4).

**Bài 2.** (1) Tổng 12, nửa 6. (2) Từ 0: tích lũy 0,1,2,12 ⇒ đỉnh 3. (3) Từ 3: tích lũy 0,10 ⇒ đỉnh 2.

**Bài 3.** (1) `du = dep[u] − dep[l]`. (2) `i ≤ du` ⇒ `kth_ancestor(u, i)`. (3) Ngược lại `kth_ancestor(v, du + dv − i)`.

**Bài 4.** (1) `len = du + dv`. (2) Chỉ số đầu tiên `i` với `2i ≥ len` là `⌈len/2⌉`. (3) Dùng `kth_on_path`.

**Bài 5.** (1) Ứng viên: đỉnh trung vị `x` và đỉnh liền trước nó trên đường. (2) So `|2s − tot|`. (3) Hòa thì chọn đỉnh liền trước (gần `u`).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
a1 = Solution().findMedian(5, E3, [[2, 3], [4, 4], [0, 4]])
a2 = Solution().findMedian(4, [[0, 1, 1], [1, 2, 1], [2, 3, 10]], [[0, 3], [3, 0]])
assert a1 == median_brute(5, E3, [[2, 3], [4, 4], [0, 4]]) == [0, 4, 2]
assert a2 == [3, 2]
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: [0, 4, 2] | Bài 2: [3, 2]


In [24]:
# Bài 3, 4
def kth_on_path(up, dep, u, v, i):
    l = lca(up, dep, u, v)
    du, dv = dep[u] - dep[l], dep[v] - dep[l]
    if i <= du:
        return kth_ancestor(up, u, i)
    return kth_ancestor(up, v, du + dv - i)

def median_by_edges(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    up = build_up(par)
    out = []
    for u, v in queries:
        l = lca(up, dep, u, v)
        L = dep[u] + dep[v] - 2 * dep[l]
        out.append(kth_on_path(up, dep, u, v, (L + 1) // 2))
    return out

random.seed(41)
for _ in range(300):
    n = random.randint(2, 12)
    e = rand_tree(n)
    par, dep, D = bfs_root(n, e)
    up = build_up(par)
    u, v = random.randrange(n), random.randrange(n)
    p = path_nodes(par, dep, u, v)
    for i in range(len(p)):
        assert kth_on_path(up, dep, u, v, i) == p[i]
    unit = [[a, b, 1] for a, b, _ in e]
    assert median_by_edges(n, e, [[u, v]]) == median_brute(n, unit, [[u, v]])
print('Bài 3 & 4: khớp vét cạn')

Bài 3 & 4: khớp vét cạn


In [25]:
# Bài 5
def closest_half(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    med = Solution().findMedian(n, edges, queries)
    out = []
    for (u, v), x in zip(queries, med):
        p = path_nodes(par, dep, u, v)
        l = min(p, key=dep.__getitem__)
        tot = D[u] + D[v] - 2 * D[l]
        def acc(node):
            idx = p.index(node)
            return D[u] - D[node] if idx <= p.index(l) else D[u] - 2 * D[l] + D[node]
        i = p.index(x)
        best = x
        if i > 0:
            y = p[i - 1]
            if abs(2 * acc(y) - tot) <= abs(2 * acc(x) - tot):
                best = y
        out.append(best)
    return out

def closest_half_brute(n, edges, queries):
    par, dep, D = bfs_root(n, edges)
    out = []
    for u, v in queries:
        p = path_nodes(par, dep, u, v)
        l = min(p, key=dep.__getitem__)
        tot = D[u] + D[v] - 2 * D[l]
        vals = []
        for idx, node in enumerate(p):
            s = D[u] - D[node] if idx <= p.index(l) else D[u] - 2 * D[l] + D[node]
            vals.append((abs(2 * s - tot), idx, node))
        out.append(min(vals)[2])
    return out

random.seed(42)
for _ in range(300):
    n = random.randint(2, 10)
    e = rand_tree(n)
    q = [[random.randrange(n), random.randrange(n)] for _ in range(5)]
    assert closest_half(n, e, q) == closest_half_brute(n, e, q)
print('Bài 5: ví dụ', closest_half(5, E3, [[3, 4], [4, 3]]))

Bài 5: ví dụ [0, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
BFS từ 0: par, dep, D;  up[0] = par (par[0] = 0);  up[j][x] = up[j-1][up[j-1][x]]
l = lca(u, v);  tot = D[u] + D[v] - 2*D[l]
tot == 0                -> u
2*(D[u]-D[l]) >= tot    -> x = u; nhảy tới y nếu dep[y] >= dep[l] và 2*(D[u]-D[y]) < tot; trả par[x]
ngược lại               -> x = v; nhảy tới y nếu dep[y] >  dep[l] và 2*(D[u]-2D[l]+D[y]) >= tot; trả x
```

**Checklist:**

- [ ] Cạnh vô hướng, BFS từ gốc 0?
- [ ] So sánh `2·s ≥ tot` (không chia nguyên)?
- [ ] Phân biệt hai nửa bằng tổng tại LCA?
- [ ] Nửa sau chỉ nhảy tới đỉnh **ngặt dưới** LCA?
- [ ] Trường hợp `u = v`?
- [ ] Không đệ quy, chạy thử `n = q = 10^5`?